# Multi-Series Insert with `td.write()`

`td.write()` inserts data for multiple series at once from a long-format DataFrame — one call instead of looping over series. This is the preferred pattern for production pipelines where data for many series arrives together.

In [1]:
try:
    import google.colab
    import urllib.request
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/rebase-energy/timedb/main/examples/colab_setup.py',
        '/tmp/colab_setup.py'
    )
    exec(open('/tmp/colab_setup.py').read())
except ImportError:
    pass

In [2]:
from datetime import datetime, timezone, timedelta
import polars as pl
import timedb as td

base_time = datetime(2025, 1, 1, tzinfo=timezone.utc)
turbines = [f"T0{i}" for i in range(1, 6)]

## Setup

`create_series_many()` registers all 5 turbines at once. The shared DataFrame uses a long format — one row per `(turbine, valid_time)` — which maps directly to `td.write()`'s expected shape.

In [3]:
td.delete()
td.create()

ids = td.create_series_many([
    {"name": "wind_power", "unit": "MW", "labels": {"turbine": t}, "overlapping": True}
    for t in turbines
])
print(f"✓ Created {len(ids)} series: {ids}")

Creating database schema...
  ✓ PostgreSQL: series_table
  ✓ ClickHouse: batches_table, flat, overlapping_short/medium/long
✓ Schema created successfully
✓ Created 5 series: [1, 2, 3, 4, 5]


In [4]:
n = 24
df = pl.DataFrame({
    "name":     ["wind_power"] * (n * len(turbines)),
    "turbine":  [t for t in turbines for _ in range(n)],
    "valid_time": pl.Series(
        [base_time + timedelta(hours=h) for _ in turbines for h in range(n)]
    ).cast(pl.Datetime("us", "UTC")),
    "value": [4.0 + i * 0.1 + j * 0.5 for j in range(len(turbines)) for i in range(n)],
})
print(f"✓ {len(df)} rows prepared")

✓ 120 rows prepared


## Pattern 1: Broadcast `knowledge_time`

All rows share the same `knowledge_time` kwarg.

In [5]:
kt = base_time - timedelta(hours=6)

results = td.write(
    df,
    name_col="name",
    label_cols=["turbine"],
    knowledge_time=kt,
    unit="MW",
)
print(f"✓ {len(results)} batches: {[str(r.batch_id)[:8] + '...' for r in results]}")

✓ 5 batches: ['019d2a18...', '019d2a18...', '019d2a18...', '019d2a18...', '019d2a18...']


## Pattern 2: Per-Row `knowledge_time`

Include a `knowledge_time` column to assign a different value per row.

In [6]:
df2 = df.with_columns(
    pl.lit(base_time - timedelta(hours=3), dtype=pl.Datetime("us", "UTC")).alias("knowledge_time")
)

results = td.write(df2, name_col="name", label_cols=["turbine"], unit="MW")
print(f"✓ {len(results)} batches")

✓ 5 batches


## Pattern 3: Multiple Batches per Call

`batch_cols` groups rows into separate batches. Useful when one DataFrame contains multiple forecast runs.

In [7]:
run_dates = [base_time - timedelta(days=2), base_time - timedelta(days=1)]

df3 = pl.concat([
    df.with_columns(pl.lit(run_date, dtype=pl.Datetime("us", "UTC")).alias("run_date"))
    for run_date in run_dates
])

results = td.write(
    df3,
    name_col="name",
    label_cols=["turbine"],
    batch_cols=["run_date"],
    unit="MW",
)
print(f"✓ {len(results)} batches (one per turbine × run_date)")

✓ 10 batches (one per turbine × run_date)


## Pattern 4: Unit Conversion

Pass `unit` to automatically convert incoming values to the series' canonical unit (MW).

In [8]:
df_kw = df.with_columns((pl.col("value") * 1000).alias("value"))

results = td.write(
    df_kw,
    name_col="name",
    label_cols=["turbine"],
    knowledge_time=base_time,
    unit="kW",
)
print(f"✓ {len(results)} batches (kW → MW conversion applied)")

✓ 5 batches (kW → MW conversion applied)


## Read Back

`list_batches()` returns provenance metadata for every insert — useful for auditing which batch contributed which data.

In [9]:
ts = td.get_series("wind_power").where(turbine="T01").read()
print(ts)

batches = td.get_series("wind_power").where(turbine="T01").list_batches()
print(f"\n✓ {len(batches)} batches for T01")

TimeSeries
┌────────────────────────────────────────┐
│  Name:             wind_power          │
│  Shape:            SIMPLE              │
│  Rows:             24                  │
│  Timezone:         UTC                 │
│  Unit:             MW                  │
│  Timeseries type:  OVERLAPPING         │
│  Labels:           {'turbine': 'T01'}  │
├────────────────────────────────────────┤
│                    wind_power          │
│  2025-01-01 00:00         4.0          │
│  2025-01-01 01:00         4.1          │
│  2025-01-01 02:00         4.2          │
│  ...                      ...          │
│  2025-01-01 21:00         6.1          │
│  2025-01-01 22:00         6.2          │
│  2025-01-01 23:00         6.3          │
└────────────────────────────────────────┘

✓ 5 batches for T01
